In [ ]:
# IMPORTO LAS LIBRERÍAS NECESARIAS
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier
import pickle

In [ ]:
# CARGO LOS DATOS
url = "../data/processed/diabetes_tree.csv"
total_data = pd.read_csv(url)
total_data

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1


In [ ]:
target_col = 'Outcome'
mi_x = total_data.drop(columns=[target_col])
mi_y = total_data[target_col]

In [ ]:
# TRAIN / TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(mi_x, mi_y, test_size=0.2, random_state=42, stratify=mi_y)

In [ ]:
config_data = {
    "n_estimators": 200,
    "max_depth": 7,
    "random_state": 42,
    "class_weight": "balanced",
    "min_samples_split": 5
}
model_out = RandomForestClassifier(**config_data)

In [ ]:
umbral = 0.65
model_out.fit(X_train, y_train)
probs_out = model_out.predict_proba(X_test)[:, 1]
predictions_out = (probs_out >= umbral).astype(int)
report_out = classification_report(y_test, predictions_out)
print(report_out)

              precision    recall  f1-score   support

           0       0.77      0.88      0.82       100
           1       0.70      0.52      0.60        54

    accuracy                           0.75       154
   macro avg       0.74      0.70      0.71       154
weighted avg       0.75      0.75      0.74       154



In [ ]:
param = {
        'criterion': ["gini", "entropy", "log_loss"],
            'max_depth': [3, 5, 7, 10, None],
                'min_samples_split': [2, 5, 10],
                    'min_samples_leaf': [1, 2, 4]
                    }

grid = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid=param, cv=5, scoring="accuracy")
grid.fit(X_train, y_train)

print("Mejores parámetros:", grid.best_params_)

Mejores parámetros: {'criterion': 'entropy', 'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 2}


In [ ]:
param_rf = {
    'n_estimators': [100, 200, 300],          # Numero de arboles en el bosque
    'criterion': ["gini", "entropy"],
    'max_depth': [3, 5, 7, 10],
    'min_samples_split': [2, 5, 10],
    'max_features': ["sqrt", "log2"],
    'class_weight':["balanced",{0:1,1:3},{0:1,1:4},{0:1,1:6}]
    # Cuantas variables ve cada arbol
}
# 2. Configuramos el GridSearch
# n_jobs=-1 usara todos los nucleos de tu procesador para ir mas rapido
grid_rf = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42), 
    param_grid=param_rf, 
    cv=5, 
    scoring="accuracy",
    n_jobs=-1 
)